In [192]:
# 1.First implementation of the Markov model - 1st order Markov model

def build_markov_model(markov_model, new_text):
    """
    Function to build or add to a 1st order Markov model given a string of text.

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
        new_text (str): a string to build or add to the moarkov_model

    Returns:
        markov_model (dict of dicts): an updated markov_model
    """

    # 1. Split the string into a list of individual words
    words = new_text.split()

    # 2. Add the artificial start and end states to the list
    words = ['*S*'] + words + ['*E*']

    # 3. Loop through the list to pair each word with the one that follows it
    for i in range(len(words) - 1):
        current = words[i]
        next_word = words[i + 1]

        # 4. If the current word isn't in our dictionary yet, add it
        if current not in markov_model:
            markov_model[current] = {}

        # 5. Count the transition to the next word
        if next_word not in markov_model[current]:
            markov_model[current][next_word] = 1
        else:
            markov_model[current][next_word] += 1

    # 6. Explicitly return the updated dictionary so it isn't 'None'
    return markov_model

In [193]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print (markov_model)

{'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


In [194]:
# 2.Second implementation of the Markov model - Nth order Markov model

def build_markov_model(markov_model, text, order=1):
    '''
    Function to build or add to a Nth order Markov model given a string of text

    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
            or None if a new model is being built
        new_text (str): a string to build or add to the moarkov_model
        order (int): the number of previous states to consider for the model

    Returns:
        markov_model (dict of dicts): an updated/new markov_model
    '''
    # 1. Initialize the dictionary if it doesn't exist yet
    if markov_model is None:
        markov_model = {}

    # 2. Split the string into a list of individual words
    words = text.split()

    # 3. Add the artificial start and end states
    # Multiply the start marker by the 'order' so the initial history window is full
    words = (['*S*'] * order) + words + ['*E*']

    # 4. Loop through the list of words
    # Stop at (len(words) - order) so we don't go out of bounds when grabbing next_word
    for i in range(len(words) - order):

        # 5. Create a tuple of the current window of words (length = order)
        current_state = tuple(words[i : i + order])

        # 6. The next word is the word immediately following the window
        next_word = words[i + order]

        # 7. If the current state isn't in our dictionary yet, add it
        if current_state not in markov_model:
            markov_model[current_state] = {}

        # 8. Count the transition to the next word
        if next_word not in markov_model[current_state]:
            markov_model[current_state][next_word] = 1
        else:
            markov_model[current_state][next_word] += 1

    # 9. Return the updated dictionary
    return markov_model

In [201]:
markov_model = dict()
text = "one fish two fish red fish blue red fish blue"
#Alternative calling code text1 that produce output that matched given expected output
text1 = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text, order=2)
markov_model


{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 2},
 ('fish', 'blue'): {'red': 1, '*E*': 1},
 ('blue', 'red'): {'fish': 1}}

In [196]:
# 3.Generate a sequence of text from the updated Nth Markov model above
import numpy as np

def get_next_word(current_word, markov_model, seed=42):
    '''
    Function to randomly move a valid next state given a markov model
    and a current state (word)

    Args:
        current_word (tuple): a word that exists in our model
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        next_word (str): a randomly selected next word based on transition probabilies

    Pseudocode:
        Calculate transition probilities for all next states from a given state (counts/sum)
        Randomly draw from these to generate the next state

    '''
    #1. get dictionary of {next_word: frequency} for current state
    current = markov_model[current_word]

    #2. total number of observed transitions out of this state
    total = sum(current.values())

    #3. get next word possibilities
    next_words = list(current.keys())
    probabilities = []

    #4. get probabilities for each transition
    for word in next_words:
        probabilities.append(current[word] / total)

    #5. randomly choose a word according wot probabilities
    next_predict = np.random.choice(next_words, p=probabilities)

    #6. return word chosen
    return next_predict

def generate_random_text(markov_model, seed=42):
    '''
    Function to generate text given a markov model

    Args:
        markov_model (dict of dicts): a dictionary of state:(next_word:frequency pairs)

    Returns:
        sentence (str): a randomly generated sequence given the model
    '''
    #1. define start and end tokens
    start, end = '*S*', '*E*'
    #2. get order number
    order = len(next(iter(markov_model)))

    #3. set the global RNG seed once
    np.random.seed(seed)

    #4. set number of start tokens to order number
    current_word = (start,) * order
    sentence = []

    #5. generate words one at a time until end token
    while True:
        next_word = get_next_word(current_word, markov_model)  # seed not passed — already seeded globally

        # end token reached
        if next_word == end:
            break

        # record generated word
        sentence.append(next_word)

        # slide the window, drop the oldest word and append the new one
        current_word = current_word[1:] + (next_word,)

    #6. combine words to final sequence
    return " ".join(sentence)


In [197]:
# 4. All the Fish
# Implementation of the Markov model for the whole book
# , includes but not limited to using the three fucntions above

In [198]:
markov_model = dict()
with open("data/one_fish_two_fish.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip blank lines
            markov_model = build_markov_model(markov_model, line)
print(sum(markov_model[("*S*",)].values())) #sanity check, expect 179
print(generate_random_text(markov_model, seed=7))

179
Some are everywhere.


In [199]:
# 5. Pick Your Poison
# Implementation of the Markov model for one of the three book given
# , includes but not limited to using the three fucntions above
poison_markov_model = dict()

with open("data/sonnets.txt", "r", encoding="utf-8") as poison_text:
    corpus = poison_text.read() # read the entire file into one big string

# seperate string into sonnet chunks and remove empty ""
sonnets = []
raw_chunks = corpus.split("\n\n")
for sonnet_chunk in raw_chunks:
    if sonnet_chunk.strip():
        sonnets.append(sonnet_chunk.strip())

# break sonnet_chunk into individual lines splitting wherever it finds a \n,
# and then glue them together, this is flattened, single-line version of the sonnet to be used for model training.
for sonnet in sonnets:
    sonnet_text = " ".join(sonnet.splitlines())

    poison_markov_model = build_markov_model(poison_markov_model, sonnet_text, order=2)

print (generate_random_text(poison_markov_model,seed=7))

When most I wink, then do mine eyes best see, For all the treasure of his spring; For such a counterpart shall fame his wit, Making his style admired every where. Give my love that still, And you in Grecian tires are painted new: Speak of the east, Nor that full star that ushers in the world is grown so bad, Mad slanderers by mad ears believed be. That I have no end: Mine appetite I never saw that you were when first I hallow'd thy fair flower add the rank smell of weeds: But why of two oaths' breach do I find, Happy to have what thou dost review The very part was consecrate to thee: 'Thou single wilt prove none.'
